### Pipeline de procesamiento de groundtruths de Docile

In [8]:
import json


def clean_annotations(json_path):

    with open(json_path, "r") as f:
        d = json.load(f)

    clean_d = {"line_item_extractions": d.get("line_item_extractions", [])}

    line_items = clean_d.get("line_item_extractions", [])
    clean_annotation = [
        {"fieldtype": item.get("fieldtype"), "text": item.get("text")}
        for item in line_items
    ]

    return {"extractions": clean_annotation}


def change_fieltypes(clean):
    extractions = []
    n = 0

    for element in clean["extractions"]:
        dicts = list(element.items())

        for element in dicts:
            if element[0] == "fieldtype":
                if element[1] == "line_item_date":
                    extractions.append({"fieldtype": "date"})
                elif element[1] == "line_item_quantity":
                    extractions.append({"fieldtype": "quantity"})
                elif element[1] == "line_item_amount_gross":
                    extractions.append({"fieldtype": "price_gross"})
                elif element[1] == "line_item_unit_price_gross":
                    extractions.append({"fieldtype": "unit_price_gross"})
                elif element[1] == "line_item_position":
                    extractions.append({"fieldtype": "position"})
                elif element[1] == "line_item_code":
                    extractions.append({"fieldtype": "code"})
            if element[0] == "text":
                extractions[n]["text"] = element[1]
        n += 1

    return extractions

In [9]:
json_path = "../data/docile/annotations/b6a51fed333341f1b51586fb.json"

ground_truth = clean_annotations(json_path)
print(ground_truth)
processed = change_fieltypes(ground_truth)
print(processed)

{'extractions': [{'fieldtype': 'line_item_position', 'text': '1'}, {'fieldtype': 'line_item_date', 'text': 'FR 02/14/20'}, {'fieldtype': 'line_item_date', 'text': 'FR 02/14/20'}, {'fieldtype': 'line_item_quantity', 'text': '8'}, {'fieldtype': 'line_item_unit_price_gross', 'text': '$63.00'}, {'fieldtype': 'line_item_position', 'text': '2'}, {'fieldtype': 'line_item_date', 'text': 'FR 02/14/20'}, {'fieldtype': 'line_item_date', 'text': 'FR 02/14/20'}, {'fieldtype': 'line_item_quantity', 'text': '6'}, {'fieldtype': 'line_item_unit_price_gross', 'text': '$63.00'}, {'fieldtype': 'line_item_position', 'text': '3'}, {'fieldtype': 'line_item_date', 'text': 'FR 02/14/20'}, {'fieldtype': 'line_item_date', 'text': 'FR 02/14/20'}, {'fieldtype': 'line_item_quantity', 'text': '8'}, {'fieldtype': 'line_item_unit_price_gross', 'text': '$63.00'}, {'fieldtype': 'line_item_position', 'text': '4'}, {'fieldtype': 'line_item_date', 'text': 'FR 02/14/20'}, {'fieldtype': 'line_item_date', 'text': 'FR 02/14/20

### Modelo para generar la query, modelo de Pydantic y ground truth

In [ ]:
from pydantic import BaseModel, Field
from typing import Dict, Any

# class SubSchema(BaseModel):
#     model_config = ConfigDict(extra = "allow")


class Model_Response(BaseModel):
    query: str = Field(..., description=" SQL query in text format")
    pydantic_model: str = Field(..., description="Pydantic model for the query")
    ground_truth_json: Dict[str, Any] = Field(
        ..., description="Ground truth JSON for the query"
    )

### Llamada al LLM para obtener query, modelo de Pydantic y ground truth

In [ ]:
import os
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv

load_dotenv()

llm_url = os.getenv("LLM_URL")
llm_model = os.getenv("LLM_MODEL")
api_key = os.getenv("API_KEY")
llm_provider = os.getenv("LLM_PROVIDER")

model = init_chat_model(
    model=llm_model,
    base_url=llm_url,
    api_key=api_key,
    temperature=0,
    timeout=60,
    max_retries=3,
    model_provider=llm_provider,
)

structured_model = model.with_structured_output(Model_Response)


def obtain_query(structured_model, processed_ground_truth):

    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "Only answer the user's query. Do not introduce your answer, only answer what you have been asked",
            ),
            (
                "human",
                """Based on {processed_ground_truth}, which is a perfect parsed table, you must:
         1) Develop a query that would be relevant for a company (for example: retrieve all transactions of type 'X' and select from them fields 'Y' and 'Z', ensuring you do not select all available fields). The condition should not be dependant on the date.
         2) Then, build a pydantic model according to this query. It must be specific and have a description for each field. Then you should create a new pydantic model which elements are a list of the elements of the pydantic model created in 2). Values of "unit_price_gross" and "price_gross" must be a Decimal, not a string. Do not import any python dependency, only the model schema. 
         3) Create the ground truth according to the query and the pydantic model created. The value of "unit_price_gross" and "price_gross" must be numbers, not strings""",
            ),
        ]
    )

    chain = prompt | structured_model

    response = chain.invoke(
        {"processed_ground_truth": json.dumps(processed_ground_truth)}
    )

    return response

In [45]:
response = obtain_query(structured_model, processed_ground_truth=processed)

In [46]:
result = response.model_dump()
print(result["query"])
print(result["pydantic_model"])
print(result["ground_truth_json"])

SELECT position, quantity, unit_price_gross FROM transactions WHERE unit_price_gross > 30.00
class TransactionItem(BaseModel):
    """Represents a single line item in a transaction."""
    position: int = Field(..., description="The sequential number of the line item.")
    quantity: int = Field(..., description="The number of units purchased.")
    unit_price_gross: Decimal = Field(..., description="The gross price per unit.")

class TransactionList(BaseModel):
    """A collection of transaction items."""
    items: list[TransactionItem] = Field(..., description="List of transaction line items.")
{'items': [{'position': 1, 'quantity': 8, 'unit_price_gross': 63.0}, {'position': 2, 'quantity': 6, 'unit_price_gross': 63.0}, {'position': 3, 'quantity': 8, 'unit_price_gross': 63.0}]}
